In [58]:
# %% Setup
# Load packages
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Paths
input_file = r"C:\Users\Miguel\Desarrollo\TFM\data\raw\predimar_miguelgomez.dta"
file2 = r"C:\Users\Miguel\Desarrollo\TFM\data\intermediate\clinical_data.parquet"
output_file = r"C:\Users\Miguel\Desarrollo\TFM\data\intermediate\clinical_data.parquet"

# Load raw data and remove labels
df = pd.read_stata(input_file)
df2 = pd.read_parquet(file2)

In [59]:
df = df[
    [
        # SOCIODEMOGRAPHIC DATA
        "sexo",  # sex
        "edad",  # age
        "fum_0m",  # smoking status
        # CLINICAL DATA
        "code",  # patient code
        "interv",  # intervention
        "bmi_autoref_0m",  # body mass index
        "glu_new0m",  # glucose
        "diabetest2_0m",  # type 2 diabetes
        "hdl_new0m",  # HDL cholesterol
        "trig_new0m",  # triglycerides
        "hipercol_0m",  # hypercholesterolemia
        "saos_0m",  # obstructive sleep apnea
        "insurenal_0m",  # renal disease
        "fglopre_MDRD_cal_0m",  # estimated glomerular filtration rate (MDRD formula)
        "fglopre_CockroftGault_cal_0m",  # estimated glomerular filtration rate (Cockroft-Gault formula)
        "hta_0m",  # hypertension
        "epoc_0m",  # chronic obstructive pulmonary disease
        "ictus_0m",  # stroke
        "bri_0m",  # bundle branch block (left)
        "brd_0m",  # bundle branch block (right)
        "ic_v00",  # heart failure
        "mio_isq_0m",  # coronary artery disease
        "miocardiopat_0m_imp",  # myocardiopathy
        "faa_0m_new",  # antiarrhythmic drugs
        "dilat_tot4_imp",  # atrial enlargement
        "eco_diamai_0m",  # atrial diameter
        "eco_areaai_0m",  # atrial area
        "eco_volumenai_0m",  # atrial volume
        "eco_fe_0m",  # ejection fraction
        "tipofa",  # atrial fibrillation type
        "tiempo_fa_ablac",  # time between atrial fibrillation diagnosis and ablation
        "ablacion_previa",  # previous ablation
        "event_blank",  # early recurrence of atrial fibrillation
        "event18m_conkardia",  # atrial fibrillation between 3 and 18 months since ablation
        "chad2ds2vasc_v00",  # chad2ds2vasc score
        ]
]

new_names = {
    "sexo": "sex",
    "edad": "age",
    "fum_0m": "smoking_status",
    "code": "code",
    "interv": "intervention",
    "bmi_autoref_0m": "BMI",
    "glu_new0m": "glucose",
    "diabetest2_0m": "diabetes",
    "hdl_new0m": "HDL",
    "trig_new0m": "triglycerides",
    "hipercol_0m": "hypercholesterolemia",
    "saos_0m": "OSA",
    "insurenal_0m": "renal_insuf",
    "fglopre_MDRD_cal_0m": "eGFR_MDRD",
    "fglopre_CockroftGault_cal_0m": "eGFR_CG",
    "hta_0m": "hypertension",
    "epoc_0m": "COPD",
    "ictus_0m": "stroke",
    "bri_0m": "lbbb",
    "brd_0m": "rbbb",
    "ic_v00": "heart_failure",
    "mio_isq_0m": "CAD",
    "miocardiopat_0m_imp": "cardiomyopathy",
    "faa_0m_new": "AAD",
    "dilat_tot4_imp": "LA_enlargment",
    "eco_diamai_0m": "LAD",
    "eco_areaai_0m": "LAA",
    "eco_volumenai_0m": "LAV",
    "eco_fe_0m": "LVEF",
    "tipofa": "AF_type",
    "tiempo_fa_ablac": "AF_duration",
    "ablacion_previa": "previous_ablation",
    "event_blank": "ERAF",
    "event18m_conkardia": "AF_recurrence",
    "chad2ds2vasc_v00": "score_chad2ds2_vasc",
    }
df = df.rename(columns=new_names)

In [60]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 35 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   sex                   720 non-null    category
 1   age                   720 non-null    float32 
 2   smoking_status        720 non-null    category
 3   code                  720 non-null    int16   
 4   intervention          720 non-null    category
 5   BMI                   720 non-null    float32 
 6   glucose               690 non-null    float32 
 7   diabetes              720 non-null    category
 8   HDL                   646 non-null    float32 
 9   triglycerides         650 non-null    float32 
 10  hypercholesterolemia  720 non-null    category
 11  OSA                   720 non-null    category
 12  renal_insuf           720 non-null    category
 13  eGFR_MDRD             533 non-null    float32 
 14  eGFR_CG               531 non-null    float32 
 15  hypertension     

In [61]:
for column in df.columns:
    if df[column].dtype.name == "category":
        print(f"Column: {column}", f"Categories: {df[column].cat.categories}", f"Ordered: {df[column].cat.ordered}")
    else:
        print(f"Column: {column}", f"Max: {df[column].max()}", f"Min: {df[column].min()}", f"Mean: {df[column].mean()}")

Column: sex Categories: Index(['hombre', 'mujer'], dtype='str') Ordered: True
Column: age Max: 84.30000305175781 Min: 24.8799991607666 Mean: 59.74891662597656
Column: smoking_status Categories: Index(['No fumador', 'Fumador activo', 'Exfumador'], dtype='str') Ordered: True
Column: code Max: 4765 Min: 1013 Mean: 2683.322222222222
Column: intervention Categories: Index(['Control', 'Intervencion'], dtype='str') Ordered: True
Column: BMI Max: 42.757286071777344 Min: 17.531044006347656 Mean: 27.800613403320312
Column: glucose Max: 332.6700134277344 Min: 5.070000171661377 Mean: 102.15135955810547
Column: diabetes Categories: Index(['no', 'si'], dtype='str') Ordered: True
Column: HDL Max: 404.6000061035156 Min: 0.9300000071525574 Mean: 46.84789276123047
Column: triglycerides Max: 614.0 Min: 23.0 Mean: 102.56928253173828
Column: hypercholesterolemia Categories: Index(['no', 'si'], dtype='str') Ordered: True
Column: OSA Categories: Index(['no', 'si'], dtype='str') Ordered: True
Column: renal_in

In [62]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 35 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   sex                   720 non-null    category
 1   age                   720 non-null    float64 
 2   smoking_status        720 non-null    category
 3   code                  720 non-null    float64 
 4   intervention          720 non-null    category
 5   BMI                   720 non-null    float64 
 6   glucose               690 non-null    float64 
 7   diabetes              720 non-null    category
 8   HDL                   646 non-null    float64 
 9   triglycerides         650 non-null    float64 
 10  hypercholesterolemia  720 non-null    category
 11  OSA                   720 non-null    category
 12  renal_insuf           720 non-null    category
 13  eGFR_MDRD             533 non-null    float64 
 14  eGFR_CG               531 non-null    float64 
 15  hypertension     

In [63]:
for column in df2.columns:
    if df2[column].dtype.name == "category":
        print(f"Column: {column}", f"Categories: {df2[column].cat.categories}", f"Ordered: {df2[column].cat.ordered}")
    else:
        print(f"Column: {column}", f"Max: {df2[column].max()}", f"Min: {df2[column].min()}", f"Mean: {df2[column].mean()}")

Column: sex Categories: Index(['male', 'female'], dtype='str') Ordered: True
Column: age Max: 84.30000305175781 Min: 24.8799991607666 Mean: 59.74891664452023
Column: smoking_status Categories: Index(['never', 'former', 'current'], dtype='str') Ordered: True
Column: code Max: 4765.0 Min: 1013.0 Mean: 2683.322222222222
Column: intervention Categories: Index(['control', 'intervention'], dtype='str') Ordered: True
Column: BMI Max: 42.757286071777344 Min: 17.531044006347656 Mean: 27.80061167611016
Column: glucose Max: 332.6700134277344 Min: 5.070000171661377 Mean: 102.15135505164879
Column: diabetes Categories: Index(['no', 'yes'], dtype='str') Ordered: True
Column: HDL Max: 404.6000061035156 Min: 0.9300000071525574 Mean: 46.84789170744618
Column: triglycerides Max: 614.0 Min: 23.0 Mean: 102.56927692706768
Column: hypercholesterolemia Categories: Index(['no', 'yes'], dtype='str') Ordered: True
Column: OSA Categories: Index(['no', 'yes'], dtype='str') Ordered: True
Column: renal_insuf Catego

In [ ]:
# %% Configuration
print("Set paths and load modules...")
# Set paths
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
INPUT_FILE = PROJECT_ROOT / "data" / "intermediate" / "clinical_data.parquet"
OUTPUT_DIR = PROJECT_ROOT / "results" / "eda" / "clinical_features"

# Load modules
import matplotlib.pyplot as plt
import pandas as pd
from src.utils.data.wrangling.statistical_analysis import plot_stratified_numeric_distribution

Set paths and load modules...


NameError: name '__file__' is not defined